In [ ]:
from Code.NetworkExpansion import *
import networkx as nx
import matplotlib.pyplot as plt
from Code.externalize_results import *


write_outputs = False
max_generation = 80

results = dict()
filters = ['NO_GOLDFORD_FLAG & NO_BRENDA_FLAG & NO_UNCLEAR_FLAG & NO_INCOMPLETE_FLAG & ALL_MOL_FILES_FOUND',
           'NO_GOLDFORD_FLAG & NO_BRENDA_FLAG & ALL_MOL_FILES_FOUND',
           'NO_UNCLEAR_FLAG & NO_INCOMPLETE_FLAG & ALL_MOL_FILES_FOUND',
           'ALL_MOL_FILES_FOUND']
filter_names = dict(zip(filters, ['stringent', 'external flags', 'internal flags', 'lax']))
reaction_scopes = {filter_names[i]: kegg_reactions.query(i)[['left', 'right']] for i in filters}

# run network expansions
for name, reaction_scope in reaction_scopes.items():
    results[name] = expand_from_seed_set(seeds=primordial_seed_set, reactions=reaction_scope, max_gen=max_generation)
    results[name].update(define_reachable_components(seeds=primordial_seed_set, reactions=reaction_scope))

# build final networks
edges = {i: generate_edge_df(reaction_scopes[i].loc[results[i]['rns'].index]) for i in reaction_scopes}
networks = {i: nx.from_pandas_edgelist(edges[i], create_using=nx.Graph, source='left', target='right', edge_attr=None) for i in reaction_scopes}

# summarize results
def result_to_df(key: str, to_apply):
    return pd.concat({i: results[i][key].apply(to_apply) for i in results.keys()}, axis=1)
summaries = {'compound count': result_to_df(key='cps', to_apply=lambda x: x.sum()),
             'reaction count': result_to_df(key='rns', to_apply=lambda x: x.sum()),
             'chiral fraction': result_to_df(key='chiral_fraction', to_apply=lambda x: x),
             'reached compound fraction': pd.DataFrame({i: results[i]['cps'].sum()/len(results[i]['reachable_compounds']) for i in results})}

# write output files
if write_outputs:
    write_output_files(results=results, edges=edges)
[df.to_csv(f'Output/summary_{name}.csv.zip', compression='zip', encoding='utf-8') for name, df in summaries.items()]

# plot results
fig, ax = plt.subplots(len(summaries.keys()), 1, figsize=(6, 10), sharex=True, layout='compressed')
for ict, iname in enumerate(summaries.keys()):
    summaries[iname].plot(ax=ax[ict], ylabel=iname, cmap='rainbow', grid=True).legend(loc='lower right')
if write_outputs:
    plt.savefig('expansion_comparison.png', dpi=300, edgecolor='w')
else:
    plt.show()
plt.close()